# 03 - Gold: ED Trends

**Pipeline:** Silver -> Gold  
**Sources:** `silver.fact_ed_performance`, `silver.dim_hospitals`  
**Target:** `gold.ed_waittime_trends` (Delta table)  

Steps:
1. Join ED performance with hospital dimension
2. Focus on MYH0005 (4-hour departure rate)
3. Flag hospitals below 67% national target
4. Calculate WA average per time period
5. Add rolling trend using window function
6. Write gold Delta table

In [ ]:
# ------------------------------------------------------------------
# 1. Load silver tables
# ------------------------------------------------------------------
from pyspark.sql.functions import col, avg, when, round, lag, count
from pyspark.sql.window import Window

perf = spark.table("silver.fact_ed_performance")
hospitals = spark.table("silver.dim_hospitals")

print(f"ED performance records: {perf.count()}")
print(f"Hospital dimension records: {hospitals.count()}")

In [ ]:
# ------------------------------------------------------------------
# 2. Filter to 4-hour departure rate (MYH0005) and join dimensions
# ------------------------------------------------------------------
ed_4hr = perf.filter(col("measure_code") == "MYH0005")

# Join on hospital_code - more reliable than hospital_name string matching
# dim_hospitals columns: hospital_code, hospital_name, latitude, longitude,
#                         private, closed, health_service
ed_joined = ed_4hr.join(
    hospitals.select("hospital_code", "health_service", "latitude", "longitude"),
    "hospital_code",
    "left"
).select(
    ed_4hr.hospital_code,
    ed_4hr.hospital_name,
    col("health_service"),
    col("latitude"),
    col("longitude"),
    col("time_period_start"),
    col("time_period_end"),
    col("value").alias("four_hour_departure_rate")
)

print(f"Joined records: {ed_joined.count()}")
ed_joined.show(5)

In [ ]:
# ------------------------------------------------------------------
# 3. Flag below national 67% target + WA average per period
# ------------------------------------------------------------------
NATIONAL_TARGET = 67.0

# WA average per time period
period_window = Window.partitionBy("time_period_start")

ed_trends = ed_joined \
    .withColumn("below_target",
        when(col("four_hour_departure_rate") < NATIONAL_TARGET, True).otherwise(False)
    ) \
    .withColumn("wa_average",
        round(avg("four_hour_departure_rate").over(period_window), 2)
    ) \
    .withColumn("variance_from_wa_avg",
        round(col("four_hour_departure_rate") - col("wa_average"), 2)
    )

# Rolling average per hospital (last 4 periods)
hospital_window = Window \
    .partitionBy("hospital_code") \
    .orderBy("time_period_start") \
    .rowsBetween(-3, 0)

ed_trends = ed_trends.withColumn(
    "rolling_4period_avg",
    round(avg("four_hour_departure_rate").over(hospital_window), 2)
)

print(f"Hospitals below {NATIONAL_TARGET}% target:")
ed_trends.filter(col("below_target") == True) \
    .select("hospital_name", "time_period_start", "four_hour_departure_rate", "wa_average") \
    .orderBy("time_period_start", "four_hour_departure_rate") \
    .show(20)

In [ ]:
# ------------------------------------------------------------------
# 4. WA summary per period
# ------------------------------------------------------------------
wa_summary = ed_trends.groupBy("time_period_start").agg(
    count("hospital_code").alias("hospital_count"),
    round(avg("four_hour_departure_rate"), 2).alias("wa_avg_4hr"),
    count(when(col("below_target") == True, 1)).alias("hospitals_below_target")
).orderBy("time_period_start")

print("WA performance by period:")
wa_summary.show(20)

In [ ]:
# ------------------------------------------------------------------
# 5. Write gold Delta table
# ------------------------------------------------------------------
ed_trends.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.ed_waittime_trends")

print("gold.ed_waittime_trends written successfully")
print(f"Total records: {spark.table('gold.ed_waittime_trends').count()}")